# Bonus — Four More Places the LLM Earns Its Keep

**ODSC Tutorial: Spec-Driven Simulation Modeling**

The main workshop drilled into two LLM uses: code generation in Block 4 and spec-compliance review in Block 5. The abstract listed six. This bonus notebook contains the other four, each as a runnable cell with a pre-written prompt and a saved-response fallback so you can use them on your own simulation work after the workshop.

Each section follows the same pattern as Block 5: build a structured prompt, send it to the LLM, fall back to a saved response if no API key is set, render the output. The saved responses were captured from real LLM runs on these prompts. They are not synthetic.

The four uses, in order:

1. **Logic review.** Function-by-function explanation of generated code, to compare against your mental model.
2. **Test scenario generation.** Adversarial what-if scenarios beyond the obvious ones, to stress the simulation's structure.
3. **Stakeholder summary.** A one-page plain-English version of the results, suitable for the decision-maker.
4. **Assumption documentation.** Extracted from the spec, reformatted for non-technical readers, organized by leverage.

## Setup

The shared helper that calls the LLM if a key is set in the environment, and otherwise falls back to the saved response.

In [1]:
import json
import os
import textwrap
from pathlib import Path

ROOT = Path("..").resolve()
TEMPLATE_A_PATH = ROOT / "templates" / "coffee_shop_des_template_a_completed.md"
CANONICAL_CODE_PATH = ROOT / "code" / "coffee_shop_des.py"
RESPONSES_DIR = Path("llm_responses")


def _read_text(p: Path) -> str:
    if not p.exists():
        raise FileNotFoundError(f"Expected to find {p}.")
    return p.read_text()


def call_llm_live(prompt: str, response_format: str = "json") -> object:
    """Call Anthropic's API. Returns parsed JSON dict if response_format=='json',
    raw string if response_format=='text'."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        raise RuntimeError("ANTHROPIC_API_KEY not set")
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    msg = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=4096,
        messages=[{"role": "user", "content": prompt}],
    )
    text = msg.content[0].text
    if response_format == "text":
        return text
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text)


def call_llm_with_fallback(prompt: str, fallback_path: Path, response_format: str = "json"):
    """Try live; fall back to saved response. Returns (response, source)."""
    try:
        return call_llm_live(prompt, response_format=response_format), "live_api"
    except Exception as e:
        print(f"[fallback] live call unavailable ({type(e).__name__}: {e}); using saved response")
        if response_format == "json":
            return json.loads(fallback_path.read_text()), "saved_fallback"
        return fallback_path.read_text(), "saved_fallback"


# Load shared inputs once.
template_a_md = _read_text(TEMPLATE_A_PATH)
canonical_code = _read_text(CANONICAL_CODE_PATH)
print(f"Template A: {len(template_a_md):,} chars")
print(f"Canonical SimPy code: {len(canonical_code):,} chars")

Template A: 15,748 chars
Canonical SimPy code: 26,362 chars


## 1. Logic review — explain what each function does

When an LLM generates code from a spec, you want to verify the code does what *you* think it does, not what the *LLM* thinks it does. The fastest way to surface a mismatch is to ask the LLM to walk through each function in plain English. If its explanation differs from your mental model, one of you is wrong, and you have a place to dig.

This is most valuable when:

- You are new to a library (SimPy in this case) and want a sanity check on idiomatic patterns.
- The generated code uses a non-obvious pattern (mutating `_capacity` here is non-standard SimPy, and worth flagging).
- You want a hand-off document for a colleague who will inherit the code.

In [2]:
LOGIC_REVIEW_PROMPT = """
You are reading a Python module that implements a discrete event simulation in SimPy. Walk through each top-level function and explain what it does in plain English. Group your explanation as JSON with these fields per function: `name`, `purpose` (one sentence), `how` (two to four sentences on the implementation), and `watch_for` (any non-obvious behavior, gotcha, or version-sensitivity).

Begin with a `clarifying_notes_first` array — at most three items the reader needs to know before reading any function (the time frame, what the central data structure represents, etc.).

End with `follow_up_questions_to_ask_the_llm` — three questions the reader might pose to deepen their understanding.

Respond with a single JSON object. Do not include any prose outside the JSON.

# CODE TO REVIEW

```python
{code}
```
""".strip()

prompt = LOGIC_REVIEW_PROMPT.format(code=canonical_code)
print(f"Prompt length: {len(prompt):,} chars")

Prompt length: 27,172 chars


In [3]:
review, source = call_llm_with_fallback(prompt, RESPONSES_DIR / "logic_review.json")
print(f"Source: {source}")
print(f"Functions covered: {len(review.get('function_walkthrough', []))}")
print()

if review.get("clarifying_notes_first"):
    print("READ THESE FIRST")
    print("-" * 60)
    for note in review["clarifying_notes_first"]:
        print(f"  - {note}")
    print()

print("FUNCTION WALKTHROUGH")
print("=" * 60)
for fn in review.get("function_walkthrough", []):
    print(f"\n{fn['name']}")
    print(f"  Purpose: {fn['purpose']}")
    print(f"  How:     {fn['how']}")
    if fn.get("watch_for"):
        print(f"  ⚠ Watch: {fn['watch_for']}")

if review.get("follow_up_questions_to_ask_the_llm"):
    print()
    print("FOLLOW-UP QUESTIONS")
    print("-" * 60)
    for q in review["follow_up_questions_to_ask_the_llm"]:
        print(f"  - {q}")

[fallback] live call unavailable (RuntimeError: ANTHROPIC_API_KEY not set); using saved response
Source: saved_fallback
Functions covered: 11

READ THESE FIRST
------------------------------------------------------------
  - The simulation clock is in *minutes from store opening* (7 AM = t=0, 5 PM = t=600). All time arithmetic in the code is in this frame.
  - The Resource is a single SimPy Resource with capacity = number of currently-on-shift baristas. Capacity is mutated at shift boundaries; this is non-standard SimPy and is the trickiest part of the file.

FUNCTION WALKTHROUGH

_assign_order_type(rng)
  Purpose: Sample one of {drip, espresso, blended} from the 30/50/20 mix.
  How:     Draws a uniform [0,1) and walks the cumulative-probability vector. No state.
  ⚠ Watch: If you change the order mix in the spec, the cumulative-probabilities are computed once at module load — restart the kernel after edits.

_draw_service_time(order_type, rng, multiplier)
  Purpose: Draw a single serv

## 2. Test scenario generation — propose stress tests

The four staffing scenarios in Template A are the obvious ones. They answer the question the owner asked. They do not test whether the simulation's *structure* holds up under stress.

A useful pattern is to ask the LLM to propose adversarial scenarios — demand shocks, resource shocks, edge cases at boundaries. For each, it states what changes, what we expect to see, and *what would invalidate the simulation* if observed. The last part is what makes this exercise valuable: each scenario is a falsifiable prediction about how the model should behave.

In [4]:
TEST_SCENARIOS_PROMPT = """
You are designing adversarial test scenarios for a discrete event simulation of a coffee shop staffing problem. The four scenarios already in the spec (Baseline, A, B, C) cover the obvious staffing levers. Your job is to propose six additional scenarios that stress the simulation's *structure* — demand shocks, resource shocks, edge cases at boundaries, behavioral extensions.

For each scenario, return JSON with: `id`, `name`, `category` (demand_shock|resource_shock|service_time_shock|behavioral_extension|sanity_check|boundary_condition), `what_changes` (the specific change from baseline, in concrete terms), `expected_behavior` (what we predict will happen), and `what_would_invalidate` (what observation, if seen in the simulation output, would tell us the model is wrong).

End with a `facilitator_note` field — a short note about which scenarios are most useful for discussion.

Respond with a single JSON object containing `context`, `scenarios`, and `facilitator_note`.

# SPECIFICATION (Template A)

{template_a}
""".strip()

prompt = TEST_SCENARIOS_PROMPT.format(template_a=template_a_md)
print(f"Prompt length: {len(prompt):,} chars")

Prompt length: 16,761 chars


In [5]:
scenarios, source = call_llm_with_fallback(prompt, RESPONSES_DIR / "test_scenarios.json")
print(f"Source: {source}")
print(f"Scenarios proposed: {len(scenarios.get('scenarios', []))}")
print()
print("CONTEXT")
print("-" * 60)
print(textwrap.fill(scenarios.get("context", ""), width=72))
print()

for s in scenarios.get("scenarios", []):
    print("=" * 72)
    print(f"[{s['id']}] {s['name']}  ({s['category']})")
    print("-" * 72)
    print("What changes:")
    print(textwrap.fill(s["what_changes"], width=72, initial_indent="  ", subsequent_indent="  "))
    print()
    print("Expected behavior:")
    print(textwrap.fill(s["expected_behavior"], width=72, initial_indent="  ", subsequent_indent="  "))
    print()
    print("What would invalidate the simulation:")
    print(textwrap.fill(s["what_would_invalidate"], width=72, initial_indent="  ", subsequent_indent="  "))
    print()

if scenarios.get("facilitator_note"):
    print("=" * 72)
    print("FACILITATOR NOTE")
    print(textwrap.fill(scenarios["facilitator_note"], width=72))

[fallback] live call unavailable (RuntimeError: ANTHROPIC_API_KEY not set); using saved response
Source: saved_fallback
Scenarios proposed: 6

CONTEXT
------------------------------------------------------------
Adversarial and edge-case scenarios beyond the four (Baseline, A, B, C)
already in Template A. Goal is to stress the model and surface places
where the structure may not hold up. Each scenario states what changes
from baseline, what we expect to see, and what would *invalidate* the
simulation if observed.

[S1] Sudden 50% morning rush spike  (demand_shock)
------------------------------------------------------------------------
What changes:
  Multiply the morning-rush arrival rate from 30/hr to 45/hr for one
  specific 30-minute window in the middle of the rush (e.g., 7:45 AM to
  8:15 AM). All other windows unchanged. Run under each staffing
  scenario.

Expected behavior:
  Walkaway rate climbs sharply — particularly under Baseline (2
  baristas) where arrival rate temporari

## 3. Stakeholder summary — write the one-pager

The simulation's audience is rarely the modeler. The decision-maker — the shop owner here, but it could be a hospital administrator, a logistics manager, an executive — wants the answer in language that connects to the decision they have to make. Translating the results into that register is its own skill, and the LLM is genuinely useful at it.

The pattern: feed in Template A and the headline results, ask for prose at a specific length and reading level, draft a one-page summary. The output is rarely shippable as-is. It is a strong starting draft you edit, not a finished memo.

In [6]:
STAKEHOLDER_SUMMARY_PROMPT = """
Write a one-page plain-English summary of a simulation analysis for the coffee shop owner. The owner is not a data scientist. They want to know what we found and what they should do about it.

Length: about 600 to 800 words. No bullet points unless absolutely necessary. No statistics jargon (no standard deviation, no p-values, no percentile notation). Use 'about 1 in 10' instead of '10th percentile'. Use 'most days' instead of 'mean'.

Structure:
- A short opening that names the question.
- What we did, in two or three sentences.
- What we found, organized by the levers tested.
- A clear recommendation, with a reason.
- An honest paragraph on what the analysis cannot tell them.

Return Markdown. Do not include any framing text outside the document itself.

# SPECIFICATION (Template A — for the question, the system, and the assumptions)

{template_a}

# HEADLINE RESULTS

{results}
""".strip()

# Headline numbers from the canonical simulation, summarised tightly.
headline_results = json.dumps({
    "baseline": {"mean_wait_min": 1.99, "p90_wait_min": 6.55, "walkaway_rate_pct": 0.9, "daily_cost_usd": 289},
    "scenario_a_rush_boost": {"mean_wait_min": 1.05, "p90_wait_min": 3.40, "walkaway_rate_pct": 0.0, "daily_cost_usd": 323},
    "scenario_b_all_day_three": {"mean_wait_min": 0.95, "p90_wait_min": 3.10, "walkaway_rate_pct": 0.0, "daily_cost_usd": 459},
    "scenario_c_process_improvement": {"mean_wait_min": 1.45, "p90_wait_min": 4.20, "walkaway_rate_pct": 0.3, "daily_cost_usd": 289},
    "morning_utilization_baseline": 0.83,
    "morning_utilization_three_baristas": 0.56,
}, indent=2)

prompt = STAKEHOLDER_SUMMARY_PROMPT.format(template_a=template_a_md, results=headline_results)
print(f"Prompt length: {len(prompt):,} chars")

Prompt length: 17,275 chars


In [7]:
summary_md, source = call_llm_with_fallback(
    prompt,
    RESPONSES_DIR / "stakeholder_summary.md",
    response_format="text",
)
print(f"Source: {source}\n")
print(summary_md)

[fallback] live call unavailable (RuntimeError: ANTHROPIC_API_KEY not set); using saved response
Source: saved_fallback

# What the simulation found, in plain English

*Prepared for: the coffee shop owner. One page. No statistics jargon.*

## The question we set out to answer

You wanted to know whether hiring a third barista during the morning rush would reduce wait times enough to be worth the extra labor cost. The short answer: it would, and the cheaper alternative — speeding up the rush workflow without adding labor — gets you most of the way there for free.

## What we did

We built a simulation of a typical day at the shop using six months of point-of-sale data. The simulation tracks every customer from the moment they arrive to the moment they leave — including the people who walk out without ordering because the line is too long. We ran each staffing scenario fifty times to see how it performs across a range of mornings, not just one.

## What we found

Today, on a normal morni

## 4. Assumption documentation — extract and reformat

Every modeler has had this conversation: "Why does the simulation show X?" "Because we assumed Y." "Wait, we *assumed* Y? Where is that written down?" Template A captures assumptions in a structured way, but the way it captures them is for the modeler, not the stakeholder. A stakeholder-facing assumption appendix should organize by *leverage* — which assumptions, if wrong, would change the answer most — and call out what is *not* in the simulation, plainly.

The pattern: feed in Template A, ask for an assumption appendix organized by leverage, with a "what is not modeled" section and a "when to rerun" section at the end.

In [8]:
ASSUMPTION_DOC_PROMPT = """
Read the simulation specification below and produce a stakeholder-facing assumption appendix in Markdown. The audience is the decision-maker, not the modeler.

Organize the appendix in this order:

1. A short opening on why assumptions matter and how to read this document.
2. Assumptions that would change the answer *most* if wrong, with a brief 'if wrong' note for each.
3. Assumptions that would change the answer *modestly* if wrong, same format.
4. Assumptions that would change the answer *least* if wrong, same format.
5. A 'what is *not* in the simulation' section listing things explicitly excluded.
6. A 'when to rerun' section listing concrete triggers that should prompt a rerun.

Use prose with light section structure. Avoid bullet-heavy formatting. The document should read like a memo, not a checklist.

Return Markdown only, no surrounding framing text.

# SPECIFICATION (Template A)

{template_a}
""".strip()

prompt = ASSUMPTION_DOC_PROMPT.format(template_a=template_a_md)
print(f"Prompt length: {len(prompt):,} chars")

Prompt length: 16,651 chars


In [9]:
assumption_md, source = call_llm_with_fallback(
    prompt,
    RESPONSES_DIR / "assumption_documentation.md",
    response_format="text",
)
print(f"Source: {source}\n")
print(assumption_md)

[fallback] live call unavailable (RuntimeError: ANTHROPIC_API_KEY not set); using saved response
Source: saved_fallback

# Modeling assumptions — for the record

*Companion to the simulation results. Plain-English version of the assumption list in Template A. Read this before relying on the simulation for a decision that costs more than the simulation cost to build.*

The simulation's answers are only as good as the assumptions it rests on. If any of these stops being true, the conclusions need to be revisited. We have organized the assumptions by how much they would change the answer if they were wrong.

## Assumptions that would change the answer most if wrong

**Customer arrivals are random and independent.** We assumed that customers arrive without coordinating with each other — one customer's decision to come in does not influence another's. This is the standard assumption behind exponential inter-arrival times and is what makes random clustering happen. *If wrong:* the simulation

## What to take from this notebook

Each cell here is a *workflow*, not a deliverable. The LLM produces a strong starting draft; you edit. The work is in the prompt design, the input curation, and the human triage step that follows. The pattern is the same in every case:

1. **Curate the input.** The spec, the code, the headline results — whichever are relevant.
2. **Constrain the output.** Specify the structure (JSON schema, document outline, length).
3. **Ask for evidence.** Make the LLM cite the input it is reasoning from.
4. **Triage.** Read what came back, keep what is right, edit what is close, discard what is wrong.

Use these prompts as templates for your own simulations. The hard part is not running the cells; it is building the spec that makes the prompts work.